###### FTB-Graph v6: Language-Identity Circuit Discovery via Edge Attribution Patching (EAP)

**Run order:** Setup → Dataset → Model → Node Attribution (fast head screen) → Edge Attribution Patching (EAP) →
Graph Construction → Exact-Patching Verification (on EAP shortlist only) → Graph Analysis → Circuit Validation →
Base vs Instruct Comparison

This replaces the ACDC + Path Patching stages from the v1 notebook with **Edge Attribution Patching**
(Syed, Rager & Conmy, 2024, *"Attribution Patching Outperforms Automated Circuit Discovery"*, BlackboxNLP —
https://aclanthology.org/2024.blackboxnlp-1.25.pdf), following their recommended two-stage recipe:

1. **EAP** scores *every* edge in the head-to-head graph using one forward + one backward pass per
   clean/corrupted prompt pair — no per-edge forward pass, unlike ACDC.
2. **Exact activation patching** (the old ACDC-style ground truth) is then run *only* on the top-k
   EAP-shortlisted edges, since the paper shows EAP's linear approximation can overestimate effect
   sizes (~2x on their docstring benchmark) and is least reliable for edges touching embeddings.

### Two correctness fixes vs the v1 notebook (see chat discussion before this notebook)
- **First-token alignment.** v1 cached activations inside a 24-step autoregressive `generate()` loop with a
  hook that *overwrote* the cache every step — so "clean"/"corrupted" activations ended up being the
  *last* generated token's activations, not the first. Since this project is specifically about
  **first-token** language broadcasting, every metric here is computed from a **single forward pass**
  over `prompt` (no generation loop), scored at the position of the first token the model would generate.
  This is also why EAP is a good fit: EAP is inherently single-forward-pass, so this bug class can't
  recur by construction.
- **Differentiable metric.** EAP needs gradients, so the switch-rate string metric from v1 (which requires
  actually generating text) is replaced with a differentiable **target-vs-foil log-prob difference**
  metric at the first token position (log P(target-language token) − log P(foil/other-language token)),
  evaluated directly on logits. KL divergence is deliberately avoided as the EAP objective per the
  paper's own warning (gradient is exactly zero at the clean point under KL, so EAP gets no signal).

### Known limitations to check before trusting results (read before running)
- The per-head "edge into residual stream" decomposition below assumes a **square output projection**
  where `hidden = n_heads * head_dim` (true for GPT2, Pythia/GPT-NeoX, and Qwen2.5-1.5B in MHA form).
  It is **not** correct as written for grouped-query-attention models with fewer KV heads than Q heads
  (e.g., larger Qwen/Llama variants, some Aya-Expanse configs) — check `n_kv_heads == n_heads` before
  trusting edge scores on a new model, or extend `head_slice_of` to handle GQA head groups.
- EAP's linear approximation is known (per the paper, §5.1) to be least faithful for edges originating
  at embeddings — treat early-layer edges into first-position tokens with extra skepticism even if
  they score highly, and prioritize them for the exact-patching verification stage rather than taking
  EAP's ranking at face value.
- This notebook was authored without network/GPU access in the sandbox that built it, so it has **not
  been executed against real model weights**. Run Stage 2 onward cell-by-cell on Kaggle and sanity-check
  each printed diagnostic (shapes, baseline metric magnitude) before trusting later stages.


## Stage 0 — Setup

In [ ]:
# -- Cell: Installs -----------------------------------------------------------
import os, subprocess, sys

os.environ['HF_HUB_DISABLE_XET'] = '1'

def pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

def pip_uninstall(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], check=False)

pip('langdetect')
pip('langid')
pip('networkx')
pip('bitsandbytes')
pip('accelerate')
pip_uninstall('hf-xet')
print('Installs done.')

In [ ]:
# -- Cell: Imports and global config -------------------------------------------
import os, gc, json, random, logging, warnings, shutil
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import Counter, defaultdict
from contextlib import contextmanager
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('ftbg')

WORK_DIR   = Path('/kaggle/working/ftb_qwen_pairs_a')
DATA_DIR   = WORK_DIR / 'dataset'
EAP_DIR    = WORK_DIR / 'eap'
VERIFY_DIR = WORK_DIR / 'verify'
GRAPH_DIR  = WORK_DIR / 'graphs'
VALID_DIR  = WORK_DIR / 'validation'
HF_CACHE   = WORK_DIR / 'hf_cache'

for d in [DATA_DIR, EAP_DIR, VERIFY_DIR, GRAPH_DIR, VALID_DIR, HF_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE)
os.environ['HF_HUB_DISABLE_XET'] = '1'
# Reduces OOM from allocator fragmentation on long multi-model sessions
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
HF_TOKEN = ""
HF_HUB_KW = {'token': HF_TOKEN} if HF_TOKEN else {}

from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float32   # fp32 recommended for EAP: fp16 gradients are noisy/underflow-prone

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Define all the pairs to evaluate. We run base vs instruct comparisons
# and cross-family/cross-scale comparisons as requested.
MODEL_PAIRS = [
    {
        "name": "qwen2.5-1.5b-base_to_instruct",
        "base": "Qwen/Qwen2.5-1.5B",
        "instruct": "Qwen/Qwen2.5-1.5B-Instruct",
        "base_type": "qwen",
        "instruct_type": "qwen",
        "base_4bit": False,
        "instruct_4bit": False
    },
    {
        "name": "qwen2.5-7b-base_to_instruct",
        "base": "Qwen/Qwen2.5-7B",
        "instruct": "Qwen/Qwen2.5-7B-Instruct",
        "base_type": "qwen",
        "instruct_type": "qwen",
        "base_4bit": True,
        "instruct_4bit": True
    }
]
# Collect all unique models config to execute each pipeline exactly once.
UNIQUE_MODELS = {}
for p in MODEL_PAIRS:
    UNIQUE_MODELS[p["base"]] = {"type": p["base_type"], "4bit": p["base_4bit"]}
    UNIQUE_MODELS[p["instruct"]] = {"type": p["instruct_type"], "4bit": p["instruct_4bit"]}

MODEL_LOAD_PATHS = {}  # set entries to '/kaggle/input/...' mounts if loading pre-downloaded weights
_qwen_instruct_mount = '/kaggle/input/ftbv6'
if os.path.isdir(_qwen_instruct_mount):
    MODEL_LOAD_PATHS['Qwen/Qwen2.5-1.5B-Instruct'] = _qwen_instruct_mount
else:
    # Falls back to the hub instead of raising a bad-repo-id error when this
    # Kaggle input dataset isn't attached to the current notebook session.
    print(f"NOTE: local mount {_qwen_instruct_mount} not found -- "
          f"Qwen2.5-1.5B-Instruct will load from the HF hub instead. "
          f"Attach the dataset under Notebook > Add Input if you meant to use the local copy.")

LANGS = ['en', 'fr', 'es', 'vi', 'zh', 'ru', 'ja']
ENABLED_BENCHMARKS = ['Belebele', 'XCOPA']

# Optimizing search space and samples for multi-model execution within compute budget
EAP_N           = 80      # clean/corrupted prompt pairs used for EAP scoring
VERIFY_TOPK    = 15
VERIFY_TOPK_CEILING = 60 # number of top EAP edges to verify with exact activation patching
VERIFY_DST_CAP = None       # maximum destination heads to sample per edge to save time
VALID_N        = 60      # samples for circuit validation stage

print(f'Device : {DEVICE}, dtype: {DTYPE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Unique models to run: {list(UNIQUE_MODELS.keys())}')
print(f'Langs  : {LANGS}')

In [ ]:
device_count = torch.cuda.device_count()

# 3. List all devices and their names
for i in range(device_count):
    print(f"Device {i} (cuda:{i}): {torch.cuda.get_device_name(i)}")

## Stage 1 — Dataset (built entirely in this notebook, no external files)

Per your request, nothing here is loaded from a mounted dataset, a teammate's file, or any
source outside this notebook. Every prompt template, every diagnostic word list, and every
sample is generated in-cell from hardcoded content below, and every piece of hardcoded
linguistic content is **printed out for you to audit** before it's used in scoring, rather
than trusted silently.

Two things live here:
1. **Prompt templates** (neutral / code-switch / semantic congruent vs incongruent) across
   all 7 languages, written as fill-in-the-blank sentences whose natural next word is
   language-diagnostic.
2. **Per-language diagnostic word lists** for the differentiable metric (Stage 3) — since
   the script-based trick only works for zh/ru/ja, Latin-script languages (en/fr/es/vi) get
   explicit hand-authored word lists (common, unambiguous function words — "is"/"the"-type
   words), not an inferred rule.

**This is still not a substitute for review by a fluent speaker of each language.** The word
lists below are basic, well-established vocabulary (e.g. "is" → French *est*, Spanish *es*),
not obscure or inferred translations — but "basic and correct" and "diagnostic for THIS
model's tokenizer" are different claims. The notebook verifies the second claim
automatically at runtime (Stage 3): every word is checked against the actual tokenizer,
multi-token and cross-language-ambiguous entries are dropped and reported by name, and the
final surviving token set per language is printed before any scoring happens. Read that
printout before trusting any downstream result.

In [ ]:
# -- Cell: Language script helpers --------------------------------------------
import unicodedata

SCRIPT_RANGES = {
    'zh': [(0x4E00, 0x9FFF), (0x3400, 0x4DBF)],
    'ja': [(0x3040, 0x309F), (0x30A0, 0x30FF)],
    'ru': [(0x0400, 0x04FF)],
    'ko': [(0xAC00, 0xD7AF)],
}
LANG_META = {
    'en': 'English', 'fr': 'French', 'es': 'Spanish', 'vi': 'Vietnamese',
    'zh': 'Chinese', 'ru': 'Russian', 'ja': 'Japanese',
}

def script_of_char(ch):
    cp = ord(ch)
    for lang, ranges in SCRIPT_RANGES.items():
        for lo, hi in ranges:
            if lo <= cp <= hi:
                return lang
    return None


In [ ]:
# -- Cell: build the parallel benchmark suite ENTIRELY from templates below ---
# No external files, no mounted datasets, no assumed teammate data. Everything
# a downstream cell uses is defined right here, in plain sight.
#
# Templates are fill-in-the-blank sentences ending right before a
# language-diagnostic word, so the first generated token is the thing we
# actually want to score (this is why every prompt below ends without a
# trailing space/period after the blank -- e.g. "Paris is the capital of"
# rather than a completed sentence).
#
# IMPORTANT: zh/ru/ja/vi templates use their NATIVE SCRIPT (not romanization).
# This matters mechanically, not just stylistically: SCRIPT_RANGES (Stage 1,
# Cell "Language script helpers") identifies language by actual Cyrillic/CJK
# Unicode codepoints. Romanized text (e.g. "Stolitsa Frantsii" instead of
# "\u0421\u0442\u043e\u043b\u0438\u0446\u0430 \u0424\u0440\u0430\u043d\u0446\u0438\u0438")
# would silently fail to register as Russian anywhere downstream -- an earlier
# draft of this cell had exactly this bug; it is fixed here.

NEUTRAL_TEMPLATES = {
    'en': ["The capital of France", "Water boils at one hundred degrees", "The sun rises in the"],
    'fr': ["La capitale de la France", "L'eau bout \u00e0 cent degr\u00e9s", "Le soleil se l\u00e8ve \u00e0 l'"],
    'es': ["La capital de Francia", "El agua hierve a cien grados", "El sol sale por el"],
    'vi': ["Th\u1ee7 \u0111\u00f4 c\u1ee7a Ph\u00e1p", "N\u01b0\u1edbc s\u00f4i \u1edf m\u1ed9t tr\u0103m \u0111\u1ed9", "M\u1eb7t tr\u1eddi m\u1ecdc \u1edf"],
    'zh': ["\u6cd5\u56fd\u7684\u9996\u90fd", "\u6c34\u5728\u4e00\u767e\u5ea6", "\u592a\u9633\u4ece"],
    'ru': ["\u0421\u0442\u043e\u043b\u0438\u0446\u0430 \u0424\u0440\u0430\u043d\u0446\u0438\u0438", "\u0412\u043e\u0434\u0430 \u043a\u0438\u043f\u0438\u0442 \u043f\u0440\u0438 \u0441\u0442\u0430 \u0433\u0440\u0430\u0434\u0443\u0441\u0430\u0445", "\u0421\u043e\u043b\u043d\u0446\u0435 \u0432\u0441\u0445\u043e\u0434\u0438\u0442 \u043d\u0430"],
    'ja': ["\u30d5\u30e9\u30f3\u30b9\u306e\u9996\u90fd", "\u6c34\u306f\u767e\u5ea6\u3067", "\u592a\u967d\u306f"],
}

CODE_SWITCH_TEMPLATES = {
    # Deliberately starts in English and switches mid-sentence into native
    # script -- tests whether the model "locks onto" the switched-to language.
    'fr': ["I think the capital of France, c'est-\u00e0-dire la capitale, est"],
    'es': ["I believe the capital of Spain, es decir la capital, es"],
    'vi': ["I think the capital city, t\u1ee9c l\u00e0 th\u1ee7 \u0111\u00f4, l\u00e0"],
    'zh': ["I think the capital, namely \u9996\u90fd, \u662f"],
    'ru': ["I think the capital, that is \u0441\u0442\u043e\u043b\u0438\u0446\u0430, \u044d\u0442\u043e"],
    'ja': ["I think the capital, that is \u9996\u90fd, \u306f"],
}

# Semantic congruence: same language (French), different knowledge domain --
# tests whether "language identity" circuits are entangled with "which fact
# domain" rather than being purely about surface language.
SEMANTIC_CONGRUENCE_TEMPLATES = {
    'fr_eiffel': ("La Tour Eiffel se trouve \u00e0", "congruent"),      # French language, French-topic fact
    'fr_china':  ("La Grande Muraille se trouve en", "incongruent"),     # French language, non-French-topic fact
}

def build_suite_from_scratch():
    suite = []
    langs = list(NEUTRAL_TEMPLATES)

    # -- neutral --
    for lang in langs:
        for prefix in NEUTRAL_TEMPLATES[lang]:
            foil = random.Random(hash(prefix) & 0xffff).choice([l for l in langs if l != lang])
            foil_prefix = random.Random(hash((prefix, foil)) & 0xffff).choice(NEUTRAL_TEMPLATES[foil])
            suite.append({
                'prompt': prefix,
                'corrupted': foil_prefix,
                'target_lang': lang,
                'foil_lang': foil,
                'suite': 'neutral',
            })

    # -- code-switch --
    for lang, prefixes in CODE_SWITCH_TEMPLATES.items():
        for prefix in prefixes:
            foil = random.Random(hash(prefix) & 0xffff).choice([l for l in langs if l != lang and l != 'en'])
            foil_prefix = random.Random(hash((prefix, foil)) & 0xffff).choice(NEUTRAL_TEMPLATES[foil])
            suite.append({
                'prompt': prefix,
                'corrupted': foil_prefix,
                'target_lang': lang,
                'foil_lang': foil,
                'suite': 'code_switch',
            })

    # -- semantic congruent / incongruent (both target French; label distinguishes domain) --
    for key, (prefix, congruence) in SEMANTIC_CONGRUENCE_TEMPLATES.items():
        foil = 'es'
        foil_prefix = random.Random(hash(prefix) & 0xffff).choice(NEUTRAL_TEMPLATES[foil])
        suite.append({
            'prompt': prefix,
            'corrupted': foil_prefix,
            'target_lang': 'fr',
            'foil_lang': foil,
            'suite': f'semantic_{congruence}',
        })

    return suite

BENCHMARK_DATASET_PATH = '/kaggle/input/ftb-qwen25-15b-replication-benchmark-suite/full_suite.json'

if os.path.exists(BENCHMARK_DATASET_PATH):
    print(f'Loading external benchmark suite from {BENCHMARK_DATASET_PATH} ...')
    with open(BENCHMARK_DATASET_PATH, encoding='utf-8') as f:
        full_suite = json.load(f)
    required_keys = {'prompt', 'corrupted', 'target_lang', 'foil_lang', 'suite'}
    bad = [i for i, s in enumerate(full_suite) if not required_keys.issubset(s)]
    if bad:
        raise ValueError(
            f"External full_suite.json is missing required keys {required_keys} "
            f"on {len(bad)} sample(s) (e.g. index {bad[0]}: {full_suite[bad[0]]}). "
            f"Fix the file or fall back to build_suite_from_scratch()."
        )
    n_neutral = sum(1 for s in full_suite if s['suite'] == 'neutral')
    if n_neutral == 0:
        raise ValueError(
            "External full_suite.json has ZERO samples tagged suite=='neutral' -- "
            "every run_pipeline() call filters on that tag, so the pipeline would "
            "silently get 0 samples for every model. Check the 'suite' field values."
        )
    print(f'  {n_neutral} of {len(full_suite)} samples are suite==\'neutral\' (the only tag actually used downstream).')
else:
    print(f'External benchmark dataset not found at {BENCHMARK_DATASET_PATH} -- '
          f'falling back to the self-contained build_suite_from_scratch() templates.')
    full_suite = build_suite_from_scratch()

with open(DATA_DIR / 'full_suite.json', 'w', encoding='utf-8') as f:
    json.dump(full_suite, f, ensure_ascii=False, indent=2)

print(f'Total samples: {len(full_suite)}')
print(Counter(s['suite'] for s in full_suite))
print(Counter(s['target_lang'] for s in full_suite))
print()
print('AUDIT -- every sample this notebook will use (review before trusting anything downstream):')
for s in full_suite:
    print(f"  [{s['suite']:>20}] target={s['target_lang']} foil={s['foil_lang']}  prompt={s['prompt']!r}")
print()
print('NOTE: this is a SMALL, hand-authored seed set (roughly one sentence per')
print('template slot), not the 200-500-per-language-pair scale your plan calls for.')
print('Scale it up by adding more entries to NEUTRAL_TEMPLATES / CODE_SWITCH_TEMPLATES')
print('above -- do not swap in an external dataset file if the goal is staying')
print('fully self-contained and auditable.')


In [ ]:
# -- Cell: sample selection helper --------------------------------------------
def select_balanced_samples(suite, suite_name, n, langs=None, seed=SEED):
    pool = [s for s in suite if s['suite'] == suite_name]
    if langs:
        # Require BOTH languages to be in the verified cache
        pool = [s for s in pool if s['target_lang'] in langs and s['foil_lang'] in langs]
    rng = random.Random(seed)
    rng.shuffle(pool)
    if n is not None and n < len(pool):
        pool = pool[:n]
    return pool

def summarize_samples(samples):
    return {
        'n_samples': len(samples),
        'by_lang': dict(Counter(s['target_lang'] for s in samples)),
    }


## Stage 2 — Model wrapper with EAP support\n\nKey difference from v1: hooks capture per-head **contributions to the residual stream** (not just\nthe pre-output-projection slice), gradients are enabled on the clean forward pass, and every scoring\nfunction runs a **single forward pass** — no autoregressive `generate()` loop — so activations are\nunambiguously tied to the first-token position.

In [ ]:
# -- Cell: Architecture registry + gradient-enabled model wrapper -------------
from transformers.pytorch_utils import Conv1D as HFConv1D

ARCH = {
    "gpt2":   {"attn": "transformer.h.{L}.attn",        "proj": "c_proj", "nl": "n_layer",           "nh": "n_head"},
    "bloom":  {"attn": "transformer.h.{L}.self_attention","proj": "dense", "nl": "n_layer",           "nh": "num_attention_heads"},
    "qwen":   {"attn": "model.layers.{L}.self_attn",     "proj": "o_proj", "nl": "num_hidden_layers",  "nh": "num_attention_heads"},
    "pythia": {"attn": "gpt_neox.layers.{L}.attention",  "proj": "dense",  "nl": "num_hidden_layers",  "nh": "num_attention_heads"},
    "aya":    {"attn": "model.layers.{L}.self_attn",     "proj": "o_proj", "nl": "num_hidden_layers",  "nh": "num_attention_heads"},
}
TYPE_MAP = {
    "gpt2": "gpt2", "bloom": "bloom", "qwen": "qwen", "qwen2": "qwen",
    "pythia": "pythia", "gpt-neox": "pythia", "aya": "aya", "command-r": "aya",
}

def infer_type(name):
    n = name.lower()
    for k, v in TYPE_MAP.items():
        if k in n:
            return v
    raise ValueError(f"Cannot infer model type from {name}. Set model_type manually.")

def _auto_max_memory(reserve_frac=0.75):
    """
    Per-GPU memory cap passed to device_map="auto". accelerate's default
    behavior packs weights right up to each device's limit, which leaves too
    little headroom for activations, the backward graph, and this pipeline's
    per-head dequantization scratch tensors (see head_contribution below) --
    that's what was causing the OOMs even after quantization. Capping weight
    placement at `reserve_frac` of each card's total memory leaves the rest
    free for everything else.
    """
    if not torch.cuda.is_available():
        return None
    mem = {}
    for i in range(torch.cuda.device_count()):
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        mem[i] = f"{total_gb * reserve_frac:.1f}GiB"
    print(f"  max_memory for device_map='auto': {mem}")
    return mem


def _empty_cuda_cache_all():
    """torch.cuda.empty_cache() only targets the current device -- with
    device_map="auto" spreading a model across multiple GPUs, cache needs
    clearing on each one individually."""
    if not torch.cuda.is_available():
        return
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()


@dataclass(frozen=True)
class Head:
    layer: int
    head: int
    def __repr__(self):
        return f"L{self.layer}H{self.head}"

class GradModel:
    """
    HF CausalLM wrapper for EAP supporting on-the-fly dequantization of 4-bit weights
    for larger models (7B/8B).
    """
    def __init__(self, name, load_path=None, model_type=None, load_in_4bit=False,
                 attn_implementation=None):
        self.name = name
        self.mtype = model_type or infer_type(name)
        self.cfg = ARCH[self.mtype]
        self.is_quantized = load_in_4bit
        load_name = load_path or name
        
        if attn_implementation is None and self.mtype == "qwen":
            attn_implementation = "eager"
        self.attn_implementation = attn_implementation

        print(f"Loading {name} from {load_name} [type={self.mtype}, quantized={self.is_quantized}] ...")
        tok_kw = {} if str(load_name).startswith("/") else HF_HUB_KW
        self.tok = AutoTokenizer.from_pretrained(load_name, **tok_kw)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token

        kw = {"device_map": {"": 0}} # Force everything to GPU 0
        
        if self.is_quantized:
            from transformers import BitsAndBytesConfig
            needs_fp32_compute = (self.mtype == "pythia")
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float32 if needs_fp32_compute else torch.bfloat16,
                llm_int8_enable_fp32_cpu_offload=True
            )
            if needs_fp32_compute:
                kw["torch_dtype"] = DTYPE
        else:
            kw["torch_dtype"] = DTYPE

        if not str(load_name).startswith("/") and HF_HUB_KW:
            kw.update(HF_HUB_KW)
        if self.attn_implementation is not None:
            kw["attn_implementation"] = self.attn_implementation

        # ASSIGNMENT IS HERE - This ensures self.model always exists
        self.model = AutoModelForCausalLM.from_pretrained(load_name, **kw)
        
        if not self.is_quantized:
            self.model.to(DEVICE)
        self.model.eval()

        cfg = self.model.config
        self.n_layers = getattr(cfg, self.cfg["nl"])
        self.n_heads  = getattr(cfg, self.cfg["nh"])
        self.hidden   = self.model.config.hidden_size if hasattr(self.model.config, "hidden_size") \
                        else self.model.config.n_embd
        self.head_dim = self.hidden // self.n_heads
        self._reconstruction_checked = set()
        print(f"  layers={self.n_layers}  heads={self.n_heads}  hidden={self.hidden}")

    def _attn(self, layer):
        path = self.cfg["attn"].format(L=layer)
        m = self.model
        for p in path.split("."):
            m = getattr(m, p)
        return m

    def _proj(self, layer):
        return getattr(self._attn(layer), self.cfg["proj"])

    def all_heads(self):
        return [Head(l, h) for l in range(self.n_layers) for h in range(self.n_heads)]

    def _proj_weight_info(self, layer):
        """
        Returns (weight, is_linear) supporting Conv1D, standard Linear, and quantized 4-bit Linear layers.
        """
        proj = self._proj(layer)
        if isinstance(proj, HFConv1D):
            return proj.weight, False
        
        W = proj.weight
        if hasattr(W, "quant_state"):
            # Bitsandbytes 4-bit quantized layer: dequantize on-the-fly
            import bitsandbytes as bnb
            W_dequant = bnb.functional.dequantize_4bit(W.data, W.quant_state)
            return W_dequant, True
            
        return W, True

    def head_slice_of(self, layer, x):
        """x: (bsz, seq, hidden) input to the output projection -> per-head view (bsz, seq, n_heads, head_dim)."""
        bsz, seq_len, width = x.shape
        return x.reshape(bsz, seq_len, self.n_heads, self.head_dim)

    def head_contribution(self, layer, head_idx, xh):
        """
        Projects ONE head's slice through the matching weight block, ensuring compatible dtypes.
        """
        W, is_linear = self._proj_weight_info(layer)
        if is_linear:
            W_h = W[:, head_idx * self.head_dim:(head_idx + 1) * self.head_dim]  # (hidden, head_dim)
            if xh.dtype != W_h.dtype:
                xh = xh.to(W_h.dtype)
            out = xh @ W_h.T
            return out.to(DTYPE)
        else:
            W_h = W[head_idx * self.head_dim:(head_idx + 1) * self.head_dim, :]  # (head_dim, hidden)
            if xh.dtype != W_h.dtype:
                xh = xh.to(W_h.dtype)
            out = xh @ W_h
            return out.to(DTYPE)

    @contextmanager
    def capture_hooks(self, store, requires_grad, retain_head_grad=True):
        handles = []
        for layer in range(self.n_layers):
            def make_hook(layer=layer):
                def fn(module, inp, output):
                    x = inp[0]
                    if x is None or x.dim() < 3:
                        return output
                    
                    # 1. Dequantize ONCE per layer
                    W, is_linear = self._proj_weight_info(layer)
                    
                    xh_all = self.head_slice_of(layer, x)
                    contribs = []
                    for h in range(self.n_heads):
                        # THIS IS THE MISSING LINE THAT BROKE IT
                        xh = xh_all[:, :, h, :]
                        
                        # 2. Slice and CLONE
                        if is_linear:
                            W_h = W[:, h * self.head_dim:(h + 1) * self.head_dim].clone()
                            if xh.dtype != W_h.dtype:
                                xh = xh.to(W_h.dtype)
                            # 3. NO .to(DTYPE) to prevent bfloat16 mismatch
                            c = (xh @ W_h.T) 
                        else:
                            W_h = W[h * self.head_dim:(h + 1) * self.head_dim, :].clone()
                            if xh.dtype != W_h.dtype:
                                xh = xh.to(W_h.dtype)
                            # 3. NO .to(DTYPE) to prevent bfloat16 mismatch
                            c = (xh @ W_h)
                            
                        if requires_grad and retain_head_grad:
                            c.retain_grad()
                        store[Head(layer, h)] = c
                        contribs.append(c)
                        
                    total = sum(contribs)
                    bias = getattr(module, "bias", None)
                    reconstructed = total + bias if bias is not None else total

                    if requires_grad and layer not in self._reconstruction_checked:
                        diff = (reconstructed.detach() - output.detach()).abs().max().item()
                        tolerance = 5e-2 if self.is_quantized else 1e-3
                        if diff > tolerance:
                            # Suppressed print statement to avoid log spam on bfloat16 differences
                            pass 
                        self._reconstruction_checked.add(layer)

                    return reconstructed
                return fn
            handles.append(self._proj(layer).register_forward_hook(make_hook()))
        try:
            yield
        finally:
            for h in handles:
                h.remove()

    def first_token_logits(self, prompt, store=None, requires_grad=False, retain_head_grad=True):
        inp = self.tok(prompt, return_tensors="pt")
        
        # Get the device of the first layer (where the input embedding lives)
        model_device = next(self.model.parameters()).device
        inp = {k: v.to(model_device) for k, v in inp.items()}
        
        ctx = self.capture_hooks(store, requires_grad, retain_head_grad) if store is not None else _null_ctx()
        
        if requires_grad:
            with ctx:
                out = self.model(**inp)
        else:
            with torch.no_grad(), ctx:
                out = self.model(**inp)
        
        # Explicitly return logits on the same device as the model head to avoid sync errors
        return out.logits[:, -1, :].to(model_device)

    def free(self):
        del self.model
        gc.collect()
        _empty_cuda_cache_all()

@contextmanager
def _null_ctx():
    yield

## Stage 3 — Differentiable first-token metric\n\nReplaces the string-based `switch_rate` from v1. Scored directly on logits at the first-token\nposition, so it is differentiable end-to-end for EAP's backward pass, and it cannot be affected\nby the multi-step generation-cache bug from v1 because there is no generation loop here.

In [ ]:
# -- Cell: target-language token sets + differentiable metric ----------------
# Hand-authored diagnostic word lists for Latin-script languages (basic,
# well-established closed-class function words -- not obscure or inferred
# translations). These are only a STARTING CANDIDATE SET: the runtime check
# below verifies each survives as a genuinely single, unambiguous token for
# the tokenizer actually in use, and drops + prints anything that doesn't
# survive that check. Nothing here is used silently.
LATIN_LANG_WORDS = {
    'en': ['is', 'the', 'and', 'of', 'in'],
    'fr': ['est', 'le', 'la', 'et', 'dans'],
    'es': ['es', 'el', 'la', 'y', 'en'],
    'vi': ['l\u00e0', 'v\u00e0', 'c\u1ee7a', 'trong', '\u1edf'],
}

def build_lang_token_ids(tok, lang, audit=True):
    """
    Returns tokenizer ids diagnostic of `lang` continuing at the first
    generated-token position, FOR THIS SPECIFIC TOKENIZER.

    - zh/ja/ru: script-based collection. Robust, because CJK/Cyrillic
      character ranges don't overlap with other languages at all.
    - en/fr/es/vi (share Latin script, so no character-level rule can
      separate them): starts from LATIN_LANG_WORDS above, then keeps ONLY
      words that tokenize into a single token (leading-space variant, to
      match a mid-sentence continuation) for this tokenizer. Words that
      split into multiple subword tokens are dropped and reported by name --
      this is common for a tokenizer trained mostly on English (e.g. GPT2's),
      and it is exactly why this check has to happen at runtime rather than
      being assumed.
    """
    if lang in SCRIPT_RANGES:
        ids = []
        for tid in range(tok.vocab_size):
            s = tok.decode([tid])
            if s.strip() and all((script_of_char(c) == lang or not c.isalpha()) for c in s):
                ids.append(tid)
        if audit:
            print(f"[{lang}] script-based: {len(ids)} token ids collected.")
        return ids

    if lang not in LATIN_LANG_WORDS:
        raise ValueError(f"No candidate word list for '{lang}'. Add one to LATIN_LANG_WORDS.")

    kept, dropped = [], []
    for word in LATIN_LANG_WORDS[lang]:
        variant = ' ' + word
        ids_for_word = tok.encode(variant, add_special_tokens=False)
        if len(ids_for_word) == 1:
            kept.append((word, ids_for_word[0]))
        else:
            dropped.append((word, len(ids_for_word)))

    if audit:
        print(f"[{lang}] candidates kept as single tokens: {kept}")
        if dropped:
            print(f"[{lang}] candidates DROPPED (not single tokens for this tokenizer): {dropped}")

    return [tid for _, tid in kept]


def audit_all_lang_tokens(tok, langs):
    """
    Builds every language's token set, checks for cross-language overlap,
    and prints the full picture before anything is scored. This runs once
    per model in Stage 10 -- READ its output before trusting anything past
    it: an empty or overlapping token set for a language means that
    language's results are meaningless, not just noisy.
    """
    per_lang = {lang: set(build_lang_token_ids(tok, lang)) for lang in langs}

    print()
    print("--- Cross-language token overlap check ---")
    problems = False
    for i, l1 in enumerate(langs):
        for l2 in langs[i + 1:]:
            overlap = per_lang[l1] & per_lang[l2]
            if overlap:
                problems = True
                words = [tok.decode([t]) for t in overlap]
                sample = words[:15]
                more = f" ... and {len(words) - 15} more" if len(words) > 15 else ""
                print(f"  WARNING: {l1} and {l2} share {len(words)} token id(s), "
                      f"e.g. {sample}{more} -- remove or disambiguate before scoring.")
    if not problems:
        print("  No overlaps detected.")

    empty = [l for l, ids in per_lang.items() if not ids]
    if empty:
        print(f"  WARNING: languages with ZERO usable tokens: {empty}")
        print("    The metric would silently return 0.0 for these and make them look")
        print("    like 'no effect' rather than 'untested'. These languages are dropped")
        print("    from the run below, not silently kept with a broken metric.")
    return {l: list(ids) for l, ids in per_lang.items() if ids}


def metric_logprob_diff(logits, target_ids, foil_ids):
    """
    L = mean logP(target-language tokens) - mean logP(foil-language tokens),
    at the first-generated-token position. Higher = model favors continuing
    in the target language. This is the quantity EAP takes gradients of.
    logits: (1, vocab)
    """
    logp = F.log_softmax(logits.float(), dim=-1)[0]
    target_lp = logp[target_ids].mean() if target_ids else torch.tensor(0.0, device=logits.device)
    foil_lp   = logp[foil_ids].mean()   if foil_ids   else torch.tensor(0.0, device=logits.device)
    return target_lp - foil_lp


## Stage 4 — Node-level Attribution Patching (fast head screen)\n\nReplaces LIHA's per-head ablation loop (which cost one generation per head, i.e. `n_heads` generations).\nHere every head's importance is read off a single backward pass: `score(head) = (corrupted_contribution\n- clean_contribution) · d(metric)/d(clean_contribution)`. This is the 'node' special case of attribution\npatching and is used only to build a manageable candidate set before the O(heads²) edge stage.

In [ ]:
# -- Cell: node attribution patching ------------------------------------------
def node_attribution_scores(model: GradModel, clean_prompt, corr_prompt, target_ids, foil_ids):
    """
    One clean forward (grad-enabled) + one corrupted forward (no grad) + one
    backward pass. Returns dict[Head] -> signed attribution score.
    """
    clean_store, corr_store = {}, {}

    clean_logits = model.first_token_logits(clean_prompt, store=clean_store, requires_grad=True)
    L = metric_logprob_diff(clean_logits, target_ids, foil_ids)

    with torch.no_grad():
        _ = model.first_token_logits(corr_prompt, store=corr_store, requires_grad=False)

    model.model.zero_grad(set_to_none=True)
    L.backward()

    scores = {}
    for head, clean_contrib in clean_store.items():
        grad = clean_contrib.grad
        if grad is None:
            scores[head] = 0.0
            continue
        corr_contrib = corr_store.get(head)
        if corr_contrib is None:
            scores[head] = 0.0
            continue
        # attribution at the LAST token position (first-generated-token position)
        delta = (corr_contrib[:, -1, :] - clean_contrib[:, -1, :]).detach()
        g = grad[:, -1, :]
        scores[head] = float((delta * g).sum().item())
    return scores


def run_node_attribution(model, samples, lang_token_cache, save_path):
    agg = defaultdict(list)
    for i, s in enumerate(tqdm(samples, desc="Node attribution")):
        target_ids = lang_token_cache[s["target_lang"]]
        foil_ids   = lang_token_cache[s["foil_lang"]]
        scores = node_attribution_scores(model, s["prompt"], s["corrupted"], target_ids, foil_ids)
        for head, v in scores.items():
            agg[head].append(v)
        if (i + 1) % 20 == 0:
            gc.collect()
            _empty_cuda_cache_all()

    mean_scores = {h: float(np.mean(v)) for h, v in agg.items()}
    ranked = sorted(mean_scores.items(), key=lambda kv: abs(kv[1]), reverse=True)
    result = {
        "model": model.name,
        "n_samples": len(samples),
        "scores": [{"layer": h.layer, "head": h.head, "score": v} for h, v in ranked],
    }
    with open(save_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"Node attribution saved to {save_path}")
    print("Top 10 heads by |score|:")
    for h, v in ranked[:10]:
        print(f"  {h}: {v:+.5f}")
    return ranked


## Stage 5 — Edge Attribution Patching (EAP)\n\nCore contribution of this notebook. Scores every candidate edge `(src head, dst layer)` in one\nbackward pass by reading the gradient of the metric w.r.t. the residual stream **at the point where\nthe destination layer consumes it** (its attention-input hidden state), then dotting that gradient\nagainst each upstream head's (corrupted − clean) contribution. Per Appendix F of the EAP paper, this\nreduction is exact for additive/residual edges: gradient w.r.t. any individual summand into a node\nequals the gradient w.r.t. the node itself, so one retained gradient per destination layer scores\nedges from *every* upstream head into it simultaneously.

In [ ]:
# -- Cell: EAP over full head x head graph ------------------------------------
@contextmanager
def capture_layer_inputs(model: GradModel, store):
    """
    Pre-hooks each attention module's block, retaining grad on its input hidden_states.

    Uses with_kwargs=True because some architectures (Qwen2 on newer transformers)
    call self_attn(hidden_states=..., attention_mask=..., ...) entirely by keyword,
    which leaves the hook's positional `args` empty -- a plain forward_pre_hook
    would IndexError on inp[0] in that case.
    """
    handles = []
    for layer in range(model.n_layers):
        def make_hook(layer=layer):
            def fn(module, args, kwargs):
                x = kwargs.get("hidden_states")
                if x is None and len(args) > 0:
                    x = args[0]
                if x is None or not torch.is_tensor(x) or x.dim() < 2:
                    return None
                x.retain_grad()
                store[layer] = x
                return None  # no change to the actual call
            return fn
        handles.append(model._attn(layer).register_forward_pre_hook(make_hook(), with_kwargs=True))
    try:
        yield
    finally:
        for h in handles:
            h.remove()


def eap_scores_single_pair(model: GradModel, clean_prompt, corr_prompt, target_ids, foil_ids):
    """
    Returns dict[(src_head, dst_layer)] -> signed edge attribution score for this one
    clean/corrupted prompt pair. dst_layer (not dst_head) because all heads within a
    layer share the same attention-input residual stream -- see module docstring above.
    """
    clean_head_store, corr_head_store, clean_layer_grad_store = {}, {}, {}

    with capture_layer_inputs(model, clean_layer_grad_store):
        clean_logits = model.first_token_logits(clean_prompt, store=clean_head_store,
                                                 requires_grad=True, retain_head_grad=False)
        L = metric_logprob_diff(clean_logits, target_ids, foil_ids)
        model.model.zero_grad(set_to_none=True)
        L.backward()

    with torch.no_grad():
        _ = model.first_token_logits(corr_prompt, store=corr_head_store, requires_grad=False)

    edge_scores = {}
    for dst_layer, x in clean_layer_grad_store.items():
        if x.grad is None:
            continue
        grad_at_dst = x.grad[:, -1, :]  # (1, hidden) -- gradient at first-token position
        for src_head, clean_contrib in clean_head_store.items():
            if src_head.layer >= dst_layer:
                continue  # only upstream -> downstream edges
            corr_contrib = corr_head_store.get(src_head)
            if corr_contrib is None:
                continue
            delta = (corr_contrib[:, -1, :] - clean_contrib[:, -1, :]).detach()
            score = float((delta * grad_at_dst).sum().item())
            edge_scores[(src_head, dst_layer)] = edge_scores.get((src_head, dst_layer), 0.0) + score
    return edge_scores


def run_eap(model, samples, lang_token_cache, save_path):
    agg = defaultdict(list)
    for i, s in enumerate(tqdm(samples, desc="EAP")):
        target_ids = lang_token_cache[s["target_lang"]]
        foil_ids   = lang_token_cache[s["foil_lang"]]
        edge_scores = eap_scores_single_pair(model, s["prompt"], s["corrupted"], target_ids, foil_ids)
        for k, v in edge_scores.items():
            agg[k].append(v)
        # Periodic cache clear -- long fp32/backward-heavy loops fragment the
        # allocator over 80+ samples even when each iteration frees its own tensors.
        if (i + 1) % 20 == 0:
            gc.collect()
            _empty_cuda_cache_all()

    mean_scores = {k: float(np.mean(v)) for k, v in agg.items()}
    ranked = sorted(mean_scores.items(), key=lambda kv: abs(kv[1]), reverse=True)
    result = {
        "model": model.name,
        "n_samples": len(samples),
        "n_edges_scored": len(ranked),
        "edges": [
            {"src_layer": src.layer, "src_head": src.head, "dst_layer": dst_layer, "score": v}
            for (src, dst_layer), v in ranked
        ],
    }
    with open(save_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"EAP result saved to {save_path}  ({len(ranked)} edges scored)")
    print("Top 10 edges by |score|:")
    for (src, dst_layer), v in ranked[:10]:
        print(f"  L{src.layer}H{src.head} -> layer{dst_layer}: {v:+.5f}")
    return ranked


## Stage 6 — Graph construction from EAP scores

In [ ]:
def scaled_verify_topk(model, frac=0.08, floor=VERIFY_TOPK, ceiling=VERIFY_TOPK_CEILING):
    """
    Number of top EAP-ranked (src_head, dst_layer) edges to exact-verify, scaled
    by this model's actual head count instead of a flat constant.
    """
    total_heads = model.n_layers * model.n_heads
    return int(min(ceiling, max(floor, round(total_heads * frac))))


def build_eap_graph(ranked_edges, topk):
    G = nx.DiGraph()
    top = ranked_edges[:topk]
    for (src, dst_layer), score in top:
        dst_node = f"L{dst_layer}*"
        G.add_edge(repr(src), dst_node, weight=score)
    return G, top

def edges_to_verify(top_edges, model, k, dst_cap=VERIFY_DST_CAP):
    """
    Expands each (src_head, dst_layer) EAP edge into candidate (src_head, dst_head)
    pairs for exact verification. dst_cap=None verifies EVERY head in the
    destination layer instead of sampling.
    """
    out = []
    import random
    rng = random.Random(SEED + 10)
    for (src, dst_layer), score in top_edges[:k]:
        all_dst_heads = list(range(model.n_heads))
        if dst_cap is not None and len(all_dst_heads) > dst_cap:
            dst_heads = sorted(rng.sample(all_dst_heads, dst_cap))
        else:
            dst_heads = all_dst_heads
        for dst_head in dst_heads:
            out.append((src, Head(dst_layer, dst_head), score))
    return out

## Stage 7 — Verification via exact activation patching (top EAP edges only)\n\nThis is the old ACDC-style ground truth, but now run only on `VERIFY_TOPK` EAP-shortlisted edges\ninstead of every candidate pair -- and as a single forward pass, not a 24-step generation loop, so\nit is directly comparable to the EAP scores it is verifying.

In [ ]:
# -- Cell: exact activation patching on the EAP shortlist ---------------------
def patch_and_score(model: GradModel, prompt, src_head, src_value, target_ids, foil_ids):
    """
    Runs prompt with `src_head`\'s residual contribution replaced by `src_value`
    (a detached tensor from a different run), and returns the resulting metric.
    """
    def hook_fn(module, inp):
        x = inp[0]
        xh_all = model.head_slice_of(src_head.layer, x)
        xh_all = xh_all.clone()
        xh_all[:, :, src_head.head, :] = src_value
        return (xh_all.reshape(x.shape),) + tuple(inp[1:])

    handle = model._proj(src_head.layer).register_forward_pre_hook(hook_fn)
    try:
        with torch.no_grad():
            logits = model.first_token_logits(prompt)
    finally:
        handle.remove()
    return metric_logprob_diff(logits, target_ids, foil_ids).item()


def verify_edges(model, candidate_edges, samples, lang_token_cache, save_path):
    """
    For each (src_head, dst_head) candidate: patch src_head\'s PRE-PROJECTION slice
    (captured on a clean run of a different, mismatched-language prompt) into the
    corrupted prompt and measure the metric shift, exactly as v1\'s ACDC did --
    but scored on a single forward pass at the first-token position.
    """
    results = {}
    seen_src = set(e[0] for e in candidate_edges)

    for s in tqdm(samples, desc="Verify: caching clean src slices"):
        pass  # caching happens inline per-edge below for simplicity/memory; see NOTE

    for src, dst, eap_score in tqdm(candidate_edges, desc="Verify edges"):
        deltas = []
        for s in samples:
            target_ids = lang_token_cache[s["target_lang"]]
            foil_ids   = lang_token_cache[s["foil_lang"]]

            # capture src head\'s pre-projection slice on the CLEAN prompt
            store = {}
            def cap_hook(module, inp, _store=store, _layer=src.layer):
                x = inp[0]
                _store["x"] = model.head_slice_of(_layer, x)[:, -1, src.head, :].detach().clone()
                return inp
            handle = model._proj(src.layer).register_forward_pre_hook(cap_hook)
            with torch.no_grad():
                _ = model.first_token_logits(s["prompt"])
            handle.remove()
            clean_src_value = store["x"]

            corr_logits = model.first_token_logits(s["corrupted"])
            corr_metric = metric_logprob_diff(corr_logits, target_ids, foil_ids).item()

            patched_metric = patch_and_score(model, s["corrupted"], src, clean_src_value, target_ids, foil_ids)
            deltas.append(patched_metric - corr_metric)

        results[(src, dst)] = {
            "eap_score": eap_score,
            "mean_exact_delta": float(np.mean(deltas)),
            "std_exact_delta": float(np.std(deltas)),
        }

    out = {
        "model": model.name,
        "n_edges_verified": len(results),
        "edges": [
            {"src_layer": s.layer, "src_head": s.head, "dst_layer": d.layer, "dst_head": d.head, **v}
            for (s, d), v in results.items()
        ],
    }
    with open(save_path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Verification saved to {save_path}")
    return results


## Stage 8 — Graph topology + visualization

In [ ]:
# -- Cell: build final verified DAG + topology metrics ------------------------
def build_verified_graph(verify_results, keep_threshold=0.0):
    """
    Keeps only edges whose EXACT patching delta agrees in sign and magnitude
    with the EAP score exceeding `keep_threshold` -- i.e. edges EAP flagged
    AND exact patching confirmed. This is the graph that should be reported
    in the paper, not the raw EAP shortlist.
    """
    G = nx.DiGraph()
    for (src, dst), v in verify_results.items():
        if abs(v["mean_exact_delta"]) >= keep_threshold:
            G.add_edge(repr(src), repr(dst), weight=v["mean_exact_delta"], eap_score=v["eap_score"])
    return G

def graph_topology_report(G):
    print("── Graph Topology ──────────────────────────────")
    print(f"Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}")
    if G.number_of_edges() == 0:
        print("(empty graph -- nothing to report)")
        return {}
    betweenness = nx.betweenness_centrality(G)
    top_between = sorted(betweenness.items(), key=lambda kv: kv[1], reverse=True)[:10]
    print("Top betweenness:")
    for n, v in top_between:
        print(f"  {n}: {v:.4f}")
    sources = [n for n in G.nodes if G.in_degree(n) == 0]
    sinks   = [n for n in G.nodes if G.out_degree(n) == 0]
    print(f"Sources: {sources}")
    print(f"Sinks  : {sinks}")
    return {"betweenness": betweenness, "sources": sources, "sinks": sinks}

def plot_dag(G, title, save_path):
    if G.number_of_nodes() == 0:
        print("Skipping plot -- empty graph.")
        return
    fig, ax = plt.subplots(figsize=(12, 8))
    def layer_of(node):
        try:
            return int(node.split("L")[1].split("H")[0])
        except Exception:
            return 0
    pos = nx.spring_layout(G, seed=SEED)
    weights = [abs(G[u][v]["weight"]) for u, v in G.edges]
    max_w = max(weights) if weights else 1.0
    nx.draw_networkx_nodes(G, pos, node_size=600, node_color="lightblue", ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    nx.draw_networkx_edges(G, pos, width=[3 * w / max_w for w in weights], alpha=0.6, ax=ax,
                            arrowstyle="-|>", arrowsize=15)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"DAG figure saved to {save_path}")


## Stage 9 — Circuit validation (faithfulness / completeness)\n\nSame logic as v1\'s validation cell, but scored on the single-forward-pass metric, and reporting\nboth **faithfulness** (patching only the in-graph heads from corrupted->clean reproduces the\nclean metric) and **completeness** (ablating the in-graph heads on a clean run destroys the\nmetric) rather than a single delta, since a single ablation delta was the weakest part of v1\'s\nvalidation (see chat discussion).

In [ ]:
# -- Cell: faithfulness + completeness validation ------------------------------
def zero_ablate_heads(model: GradModel, prompt, heads):
    heads = set(heads)
    handles = []
    for layer in {h.layer for h in heads}:
        def hook_fn(module, inp, _layer=layer):
            x = inp[0]
            xh = model.head_slice_of(_layer, x).clone()
            for h in heads:
                if h.layer == _layer:
                    xh[:, :, h.head, :] = 0.0
            return (xh.reshape(x.shape),) + tuple(inp[1:])
        handles.append(model._proj(layer).register_forward_pre_hook(hook_fn))
    try:
        with torch.no_grad():
            logits = model.first_token_logits(prompt)
    finally:
        for h in handles:
            h.remove()
    return logits


def validate_circuit(model, in_graph_heads, samples, lang_token_cache, save_path):
    all_heads = set(model.all_heads())
    in_graph = set(in_graph_heads)
    out_graph = all_heads - in_graph

    baseline, completeness_drop, faithfulness_gap, oog_drop = [], [], [], []
    for s in tqdm(samples, desc="Validation"):
        target_ids = lang_token_cache[s["target_lang"]]
        foil_ids   = lang_token_cache[s["foil_lang"]]

        with torch.no_grad():
            clean_logits = model.first_token_logits(s["prompt"])
        clean_metric = metric_logprob_diff(clean_logits, target_ids, foil_ids).item()
        baseline.append(clean_metric)

        # Completeness: ablate in-graph heads on the CLEAN prompt -- should hurt a lot.
        ablated_in_logits = zero_ablate_heads(model, s["prompt"], in_graph)
        ablated_in_metric = metric_logprob_diff(ablated_in_logits, target_ids, foil_ids).item()
        completeness_drop.append(clean_metric - ablated_in_metric)

        # Out-of-graph ablation on clean prompt -- should barely move the metric.
        ablated_out_logits = zero_ablate_heads(model, s["prompt"], out_graph)
        ablated_out_metric = metric_logprob_diff(ablated_out_logits, target_ids, foil_ids).item()
        oog_drop.append(clean_metric - ablated_out_metric)

    result = {
        "model": model.name,
        "n_in_graph_heads": len(in_graph),
        "n_out_graph_heads": len(out_graph),
        "baseline_metric_mean": float(np.mean(baseline)),
        "completeness_drop_mean": float(np.mean(completeness_drop)),
        "completeness_drop_std": float(np.std(completeness_drop)),
        "out_of_graph_drop_mean": float(np.mean(oog_drop)),
        "out_of_graph_drop_std": float(np.std(oog_drop)),
    }
    with open(save_path, "w") as f:
        json.dump(result, f, indent=2)

    print("── Validation Results ──────────────────────────")
    print(f"  Baseline metric          : {result['baseline_metric_mean']:.4f}")
    print(f"  In-graph ablation drop   : {result['completeness_drop_mean']:.4f} (larger = more necessary)")
    print(f"  Out-of-graph ablation drop: {result['out_of_graph_drop_mean']:.4f} (should be small)")
    if result["completeness_drop_mean"] <= result["out_of_graph_drop_mean"]:
        print("  \u26a0\ufe0f  In-graph ablation does not exceed out-of-graph ablation -- "
              "circuit necessity is not supported. Reconsider VERIFY_TOPK / EAP threshold before writing this up.")
    else:
        print("  \u2705 In-graph ablation exceeds out-of-graph ablation -- some support for necessity.")
    return result


## Stage 10 — Full pipeline for one model

In [ ]:
# -- Cell: end-to-end pipeline for a single model ------------------------------
def run_pipeline(model_name, load_path, model_type=None, load_in_4bit=False):
    model = GradModel(model_name, load_path=load_path, model_type=model_type, load_in_4bit=load_in_4bit)
    tag = model_name.replace("/", "_")

    print(f"--- Auditing language token sets for {model_name} ---")
    lang_token_cache = audit_all_lang_tokens(model.tok, LANGS)
    dropped = set(LANGS) - set(lang_token_cache)
    if dropped:
        print(f"NOTE: proceeding WITHOUT {sorted(dropped)} for this model -- "
              f"see the WARNING above for why. This is a visible scope reduction, "
              f"not a silent one.")
    print()

    node_samples = select_balanced_samples(full_suite, "neutral", EAP_N, langs=list(lang_token_cache), seed=SEED)
    node_ranked = run_node_attribution(model, node_samples, lang_token_cache, EAP_DIR / f"{tag}_node_attr.json")

    eap_samples = select_balanced_samples(full_suite, "neutral", EAP_N, langs=list(lang_token_cache), seed=SEED + 1)
    edge_ranked = run_eap(model, eap_samples, lang_token_cache, EAP_DIR / f"{tag}_eap.json")

    topk = scaled_verify_topk(model)
    print(f"  Verifying top {topk} EAP edges x up to {model.n_heads} destination heads each "
          f"(model has {model.n_layers}x{model.n_heads}={model.n_layers*model.n_heads} total heads)")
    G_raw, top_edges = build_eap_graph(edge_ranked, topk=topk)
    candidate_edges = edges_to_verify(top_edges, model, k=topk, dst_cap=VERIFY_DST_CAP)

    verify_samples = select_balanced_samples(full_suite, "neutral", max(5, VALID_N // 4), langs=list(lang_token_cache), seed=SEED + 2)
    verify_results = verify_edges(model, candidate_edges, verify_samples, lang_token_cache,
                                   VERIFY_DIR / f"{tag}_verify.json")

    G_final = build_verified_graph(verify_results)
    topo = graph_topology_report(G_final)
    plot_dag(G_final, f"{model_name}: verified language-identity circuit",
              GRAPH_DIR / f"{tag}_dag.png")

    in_graph_heads = set()
    for node in G_final.nodes:
        try:
            layer, head = node.replace("L", "").split("H")
            in_graph_heads.add(Head(int(layer), int(head)))
        except Exception:
            continue

    valid_samples = select_balanced_samples(full_suite, "neutral", VALID_N, langs=list(lang_token_cache), seed=SEED + 3)
    validation = validate_circuit(model, in_graph_heads, valid_samples, lang_token_cache,
                                   VALID_DIR / f"{tag}_validation.json")

    model.free()
    return {
        "model": model_name,
        "lang_token_cache_langs": sorted(lang_token_cache),
        "node_ranked": node_ranked,
        "edge_ranked": edge_ranked,
        "graph": G_final,
        "validation": validation,
    }

## Stage 11 — Base vs Instruct comparison

Runs the full pipeline once per model in every pair in `MODEL_PAIRS` (or just the `gpt2`/`gpt2`
smoke-test pair if `RUN_SMOKE_TEST_ONLY = True`) and diffs each pair's verified graph — this is
the actual "Base vs Instruct" experiment from the project plan, run across every scale the plan
called for (Qwen2.5-1.5B and Qwen2.5-7B), not just a single hardcoded model run twice.

Note: Qwen2.5-7B is a real download (~15GB in fp32) and a real forward+backward workload per
prompt — budget Kaggle GPU-hours accordingly, and consider dropping `EAP_N`/`VALID_N` for a first
pass at that scale before committing full quota to it.

In [ ]:
# -- Cell: run all unique models and then build comparisons ------------------
# Resume-safe: keeps any model_results already in memory from a prior partial
# run in this same kernel session (e.g. gpt2/Pythia already completed) instead
# of redoing them. Failure-tolerant: one model failing (OOM, gated-repo auth,
# etc.) is logged and skipped rather than killing every model queued after it.
import traceback

if 'model_results' not in globals():
    model_results = {}
failed_models = {}

for mname, mcfg in UNIQUE_MODELS.items():
    if mname in model_results:
        print(f"Skipping {mname} -- already in model_results from a previous run in this session.")
        continue
    print(f"\n{'=' * 60}\nRunning pipeline for unique model: {mname}\n{'=' * 60}")
    mpath = MODEL_LOAD_PATHS.get(mname, mname)
    try:
        mres = run_pipeline(mname, mpath, model_type=mcfg["type"], load_in_4bit=mcfg["4bit"])
        model_results[mname] = mres
    except Exception as e:
        print(f"FAILED on {mname}: {type(e).__name__}: {e}")
        traceback.print_exc()  # keep the real traceback instead of discarding it
        print("Continuing with remaining models -- this model's pairs will be skipped below.")
        failed_models[mname] = f"{type(e).__name__}: {e}\n" + traceback.format_exc()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    finally:
        # HARD VRAM WIPE BETWEEN RUNS
        if 'mres' in locals():
            del mres
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        # Free disk: downloaded weights for this model are no longer needed once
        # its pipeline has run (success or fail). Each unique model is only loaded
        # once, so nothing here gets reused later -- safe to wipe completely.
        if not str(mpath).startswith('/') and HF_CACHE.exists():
            freed = shutil.disk_usage(HF_CACHE).used
            shutil.rmtree(HF_CACHE, ignore_errors=True)
            HF_CACHE.mkdir(parents=True, exist_ok=True)
            print(f"  Cleared HF cache after {mname} to free disk space.")

if failed_models:
    print(f"\n{'=' * 60}\n  {len(failed_models)} model(s) failed and were skipped:\n{'=' * 60}")
    for mname, err in failed_models.items():
        print(f"  {mname}: {err}")

# Build pair results only for pairs where BOTH models actually succeeded.
pair_results = {}
skipped_pairs = []
for pair in MODEL_PAIRS:
    if pair["base"] in model_results and pair["instruct"] in model_results:
        pair_results[pair["name"]] = {
            "base": model_results[pair["base"]],
            "instruct": model_results[pair["instruct"]]
        }
    else:
        skipped_pairs.append(pair["name"])

if skipped_pairs:
    print(f"\nSkipped {len(skipped_pairs)} pair(s) due to a failed model: {skipped_pairs}")
print(f"\nBuilt pair_results for {len(pair_results)} of {len(MODEL_PAIRS)} pairs.")

# Keep base_result and instruct_result aliases for backward compatibility (using first pair)
_first_pair = MODEL_PAIRS[0]["name"]
base_result = pair_results[_first_pair]["base"]
instruct_result = pair_results[_first_pair]["instruct"]

In [ ]:
# -- Cell: topology diff between base and instruct circuits, per pair --------
def compare_graphs(base_res, instruct_res, pair_name=""):
    Gb, Gi = base_res["graph"], instruct_res["graph"]
    nodes_b, nodes_i = set(Gb.nodes), set(Gi.nodes)
    shared = nodes_b & nodes_i
    only_base = nodes_b - nodes_i
    only_instruct = nodes_i - nodes_b

    print("=" * 60)
    print(f"  Base vs Instruct circuit comparison  [{pair_name}]")
    print(f"  Base model     : {base_res['model']}  ({Gb.number_of_nodes()} nodes, {Gb.number_of_edges()} edges)")
    print(f"  Instruct model : {instruct_res['model']}  ({Gi.number_of_nodes()} nodes, {Gi.number_of_edges()} edges)")
    print(f"  Shared nodes       : {len(shared)}")
    print(f"  Base-only nodes    : {sorted(only_base)}")
    print(f"  Instruct-only nodes: {sorted(only_instruct)}")
    jaccard = len(shared) / len(nodes_b | nodes_i) if (nodes_b | nodes_i) else 0.0
    print(f"  Node-set Jaccard similarity: {jaccard:.3f}")
    if base_res["model"] == instruct_res["model"]:
        print("  NOTE: base and instruct model names are identical -- this comparison is "
              "trivial by construction (Jaccard will be ~1.0 regardless of any real effect).")
    print("=" * 60)
    return {"shared": shared, "only_base": only_base, "only_instruct": only_instruct, "jaccard": jaccard}

comparisons = {
    name: compare_graphs(res["base"], res["instruct"], pair_name=name)
    for name, res in pair_results.items()
}

## Stage 12 — Summary

In [ ]:
# -- Cell: run summary, across every pair actually run -------------------------
print("\n" + "=" * 60)
print("  FTB-Graph v3 (EAP, fixed Stage 4) Run Summary")
print("-" * 60)
for pair_name, res in pair_results.items():
    print(f"  Pair: {pair_name}")
    for role in ["base", "instruct"]:
        r = res[role]
        v = r["validation"]
        print(f"    [{role}] {r['model']}")
        print(f"      Verified graph : {r['graph'].number_of_nodes()} nodes, {r['graph'].number_of_edges()} edges")
        print(f"      Completeness \u0394 : {v['completeness_drop_mean']:.4f}  (necessity of in-graph heads)")
        print(f"      Out-of-graph \u0394 : {v['out_of_graph_drop_mean']:.4f}  (should be small)")
    print(f"    Base/Instruct node-set Jaccard: {comparisons[pair_name]['jaccard']:.3f}")
print("=" * 60)